In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1. Carregar os dados
df = pd.read_csv("m_30_bcts.csv")

# 2. Criar o Boxplot Interativo
fig = px.box(
    df, 
    x="modelo", 
    y="erro", 
    color="modelo",
    points="all",          # Mostra todos os pontos (datasets) ao lado do box
    hover_data=["dataset", "n_classes_original"], # Info extra ao passar o mouse
    title="Distribuição do Erro de Quantificação (MAE) por Modelo",
    labels={"erro": "Erro Médio Absoluto (MAE)", "modelo": "Algoritmo/Configuração"},
    category_orders={"modelo": ["EMQ", "MoSS_3", "MoSS_4", "MoSS_7"]} # Organiza a ordem
)

# 3. Customizar o layout para parecer um paper científico
fig.update_layout(
    template="plotly_white",
    showlegend=False,
    xaxis_title="Modelo",
    yaxis_title="Erro (MAE)",
    font=dict(family="Arial", size=14)
)

# 4. Exibir o gráfico
fig.show()

# 5. Opcional: Salvar como HTML (interativo) para abrir no navegador
#pio.write_html(fig, file='1.html', auto_open=True)

In [3]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Carregar os dados
df = pd.read_csv("m_30_bcts.csv")

# 2. Configurações de layout
datasets = df['dataset'].unique()
n_datasets = len(datasets)
# Criamos uma subfigura por dataset, em uma única coluna
fig = make_subplots(
    rows=n_datasets, cols=1, 
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02 # Espaço curto entre os gráficos
)

# 3. Iterar e adicionar cada gráfico com sua própria ordem
for i, ds in enumerate(datasets, 1):
    df_ds = df[df['dataset'] == ds].copy()
    
    # Calcular a ordem local (pela mediana do erro neste dataset)
    ordem_local = df_ds.groupby("modelo")["erro"].median().sort_values().index.tolist()
    
    # Adicionar um boxplot para cada modelo, seguindo a ordem local
    for modelo in ordem_local:
        df_mod = df_ds[df_ds['modelo'] == modelo]
        fig.add_trace(
            go.Box(
                y=df_mod['erro'],
                name=modelo,
                boxpoints='outliers',
                legendgroup=modelo,
                showlegend=(i == 1) # Só mostra a legenda no primeiro gráfico
            ),
            row=i, col=1
        )

# 4. Ajustes finais de tamanho e estética
fig.update_layout(
    height=n_datasets * 400, # 300px para cada dataset
    template="plotly_white",
    title_text="Performance Local: Modelos Ordenados do Melhor para o Pior por Dataset",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

# Deixar os eixos X independentes para cada subgráfico respeitar sua ordem
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="MAE")
#fig.write_html("meu_resultado_ordenado.html")
fig.show()

In [11]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# ============================================================
# 1. Carregar os dados
# ============================================================

CSV_GLOBAL = os.path.join("..", "exp_011", "m_30_bcts.csv")
CSV_PERCLASS = "m_30_bcts_perclass.csv"

df_global = pd.read_csv(CSV_GLOBAL)
df_perclass = pd.read_csv(CSV_PERCLASS)

# ============================================================
# 2. Normalizar nomes dos modelos
# ============================================================

df_global = df_global.copy()
df_global["modelo"] = df_global["modelo"].apply(
    lambda x: "MoSS_Global" if x.startswith("MoSS_") else x
)

df_perclass = df_perclass.copy()
df_perclass["modelo"] = df_perclass["modelo"].apply(
    lambda x: "MoSS_PerClass" if x.startswith("MoSS_") else x
)

# ============================================================
# 3. Manter EMQ_BCTS apenas uma vez
# ============================================================

df_emq = df_global[df_global["modelo"] == "EMQ_BCTS"]

df_global = df_global[df_global["modelo"] != "EMQ_BCTS"]
df_perclass = df_perclass[df_perclass["modelo"] != "EMQ_BCTS"]

# ============================================================
# 4. Concatenar tudo
# ============================================================

df = pd.concat(
    [df_global, df_perclass, df_emq],
    ignore_index=True
)

# ============================================================
# 5. CONTAGEM DE VITÓRIAS (GLOBAL vs PER-CLASS)
# ============================================================

wins = {"MoSS_Global": 0, "MoSS_PerClass": 0, "Empate": 0}

for ds in df["dataset"].unique():
    df_ds = df[df["dataset"] == ds]

    med_global = df_ds[df_ds["modelo"] == "MoSS_Global"]["erro"].median()
    med_percls = df_ds[df_ds["modelo"] == "MoSS_PerClass"]["erro"].median()

    if med_global < med_percls:
        wins["MoSS_Global"] += 1
    elif med_percls < med_global:
        wins["MoSS_PerClass"] += 1
    else:
        wins["Empate"] += 1

wins_df = pd.DataFrame.from_dict(
    wins, orient="index", columns=["Vitórias"]
)

# ============================================================
# 6. RANKING FINAL (GLOBAL × PER-CLASS × EMQ)
# ============================================================

ranking = (
    df.groupby("modelo")["erro"]
    .median()
    .sort_values()
    .reset_index()
)

ranking.columns = ["Modelo", "Mediana MAE"]

# ============================================================
# 7. Plotagem (igual ao seu)
# ============================================================

datasets = df["dataset"].unique()
n_datasets = len(datasets)

fig = make_subplots(
    rows=n_datasets,
    cols=1,
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=0.02
)

for i, ds in enumerate(datasets, start=1):
    df_ds = df[df["dataset"] == ds].copy()

    ordem_local = (
        df_ds.groupby("modelo")["erro"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    for modelo in ordem_local:
        df_mod = df_ds[df_ds["modelo"] == modelo]

        fig.add_trace(
            go.Box(
                y=df_mod["erro"],
                name=modelo,
                boxpoints="outliers",
                legendgroup=modelo,
                showlegend=(i == 1)
            ),
            row=i,
            col=1
        )

# ============================================================
# 8. Estética final
# ============================================================

fig.update_layout(
    height=n_datasets * 400,
    template="plotly_white",
    title_text="Comparação Local por Dataset — MoSS Global vs MoSS Per-Class vs EMQ-BCTS",
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True
)

fig.update_yaxes(title_text="MAE")
fig.update_xaxes(showgrid=False)

fig.write_html("comparacao_global_vs_perclass.html")
fig.show()

# ============================================================
# 9. RELATÓRIO FINAL (texto + tabelas)
# ============================================================

print("\n🏆 CONTAGEM DE VITÓRIAS (por dataset)")
print(wins_df)

print("\n📊 RANKING FINAL (mediana global do MAE)")
print(ranking)


🏆 CONTAGEM DE VITÓRIAS (por dataset)
               Vitórias
MoSS_Global          19
MoSS_PerClass         5
Empate                0

📊 RANKING FINAL (mediana global do MAE)
          Modelo  Mediana MAE
0       EMQ_BCTS     0.024204
1    MoSS_Global     0.079858
2  MoSS_PerClass     0.108102


In [7]:
# ============================================================
# CONTAGEM FINAL DE VITÓRIAS (incluindo EMQ)
# ============================================================

modelos = df["modelo"].unique()
datasets = df["dataset"].unique()

wins = {m: 0 for m in modelos}

for ds in datasets:
    df_ds = df[df["dataset"] == ds]

    # Mediana do erro por modelo neste dataset
    medians = (
        df_ds.groupby("modelo")["erro"]
        .median()
    )

    # Menor MAE vence
    best_value = medians.min()
    winners = medians[medians == best_value].index.tolist()

    # Se houver empate, todos ganham 1 ponto
    for w in winners:
        wins[w] += 1

wins_df = (
    pd.DataFrame.from_dict(wins, orient="index", columns=["Vitórias"])
    .sort_values("Vitórias", ascending=False)
)

print("\n🏆 CONTAGEM FINAL DE VITÓRIAS (por dataset)")
print(wins_df)



🏆 CONTAGEM FINAL DE VITÓRIAS (por dataset)
               Vitórias
EMQ_BCTS             21
MoSS_PerClass         3
MoSS_Global           0
